# 04 · Crop-disease vision model **v4** — EfficientNet-B2 @ 288 px, stronger training → ExecuTorch
**Bharat AI-SoC 2026–27 · PS1 · IlaiNalam**

**Goal:** read finer leaf detail so look-alike diseases are separated better (e.g. *Septoria leaf spot ↔ Leaf mold*, *Peach ↔ Orange*).
**Unchanged:** the same 38 classes, same class order, same train/val/test split → `labels.json`, `kb.json`, crop picker, leaf/spot views and severity in the app stay exactly as they are. Only the disease-naming model is replaced.

**What is stronger than v3**
| | v3 | **v4** |
|---|---|---|
| Backbone | MobileNetV3-Large | **EfficientNet-B2** (ImageNet) |
| Input | 224 px | **288 px** (small spots keep more pixels) |
| Epochs | 6 | **10**, cosine schedule with warm-up |
| Augmentation | crops, colour, blur, JPEG, background swap | same **+ TrivialAugment + CutMix/MixUp** |
| Class balance | none | **balanced sampling** (rare classes seen more often) |
| Weights kept | best epoch | **EMA** (smoothed average of weights — usually more accurate and stable) |
| Real photos | PlantDoc train ×4 | same; best epoch picked on lab + real-photo validation |
| On the phone | 1 pass | **2 passes (photo + mirror image), averaged** |

**Kaggle setup:** new notebook → *File ▸ Import Notebook* → this file · **GPU T4 x2** · **Internet On** · *Add Input*: **"plantvillage dataset"** (abdallahalidev) and **your v3 dataset** (`ps1_vision_outputs_v3` — used only to compare v3 vs v4; optional) · **Run All** (~2–2.5 h) · download **`ps1_vision_outputs_v4.zip`**.

In [ ]:
# Uses Kaggle's pre-installed torch/torchvision. Do NOT pip-install torch or executorch in this kernel.
!pip install -q --default-timeout=120 --retries 10 kagglehub opencv-python-headless scikit-learn
import os, sys, json, random, glob, time, shutil, subprocess, re, io
import numpy as np, torch, torchvision, cv2
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Accelerator to GPU T4 x2")

In [ ]:
# ---- Config -------------------------------------------------------------
SEED, IMG_SIZE, BATCH, EPOCHS, LR, WARMUP_EPOCHS = 42, 288, 48, 8, 1e-3, 1   # 8: the first run peaked on real photos at epoch 7 of 10
BG_SWAP_PROB, PLANTDOC_REPEAT, MIX_PROB, EMA_DECAY = 0.5, 4, 0.5, 0.999
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK      = "/kaggle/working" if IS_KAGGLE else "/content"
OUT_DIR   = os.environ.get("OUT_DIR_OVERRIDE", f"{WORK}/out_v4")
ENV_DIR   = os.environ.get("ET_ENV_DIR", "/tmp/et_env")
TMP       = os.environ.get("TMP_DIR", "/tmp/ps1")
FAST_TEST = os.environ.get("FAST_TEST") == "1"               # local smoke test only
if FAST_TEST: EPOCHS, BATCH = 1, 8
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(TMP, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU)
def ram():   # host memory — the first run's kernel died from it after ~2 h
    m = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo")}
    return f"RAM used {(m['MemTotal'] - m['MemAvailable']) / 1e6:.1f} / {m['MemTotal'] / 1e6:.1f} GB"
# Resume: if best.pt from an earlier run is attached as input (Add Input → your notebook's output), training is skipped.
RESUME = (glob.glob("/kaggle/input/**/out_v4/best.pt", recursive=True) or glob.glob("/kaggle/input/**/best.pt", recursive=True)) if IS_KAGGLE else []
print("resume from:", RESUME[0] if RESUME else "none — full training")
print(ram())

def sh(cmd, timeout=None, check=True):
    print("$", cmd)
    try: r = subprocess.run(cmd, shell=True, timeout=timeout).returncode
    except subprocess.TimeoutExpired: print("  …timed out"); r = -1
    if check: assert r == 0, f"command failed ({r}): {cmd}"
    return r

# v3 model for the comparison (optional)
V3 = os.environ.get("FAST_V3_DIR") if FAST_TEST else None
if not V3:
    hits = glob.glob("/kaggle/input/**/crop_disease_int8.pte", recursive=True)
    for z in ([] if hits else glob.glob("/kaggle/input/**/*.zip", recursive=True)):
        shutil.unpack_archive(z, f"{TMP}/v3"); hits = glob.glob(f"{TMP}/v3/**/crop_disease_int8.pte", recursive=True)
        if hits: break
    V3 = os.path.dirname(hits[0]) if hits else ""
print("v3 model for comparison:", V3 or "not attached (comparison skipped)")

## 1 · PlantVillage — same classes, same split as v2

In [ ]:
if FAST_TEST:
    DATA_DIR = os.environ["FAST_DATA_DIR"]
else:
    cands = [p for p in glob.glob("/kaggle/input/**/color", recursive=True) if os.path.isdir(p)]
    if not cands:
        import kagglehub
        root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
        cands = [p for p in glob.glob(os.path.join(root, "**", "color"), recursive=True) if os.path.isdir(p)]
    assert cands, "PlantVillage 'color' folder not found — add the dataset via Add Input"
    DATA_DIR = cands[0]
print("DATA_DIR:", DATA_DIR)

classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
cls_idx = {c: i for i, c in enumerate(classes)}
def parse_label(raw):
    crop, _, cond = raw.partition("___")
    clean = lambda s: s.replace("_", " ").replace(",", "").strip()
    return {"raw": raw, "crop": clean(crop), "condition": clean(cond) or "unknown", "healthy": cond.lower() == "healthy"}
labels = [parse_label(c) for c in classes]
json.dump(labels, open(f"{OUT_DIR}/labels.json", "w"), indent=1)
crop_of = [l["crop"] for l in labels]
print(len(classes), "classes")

from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, Dataset
pv_all = ImageFolder(DATA_DIR)
assert pv_all.classes == classes
idx = list(range(len(pv_all))); random.Random(SEED).shuffle(idx)          # identical to v2
n_val = n_test = int(0.1 * len(idx))
val_idx, test_idx, train_idx = idx[:n_val], idx[n_val:n_val + n_test], idx[n_val + n_test:]
json.dump({"data_dir": DATA_DIR, "classes": classes, "train": train_idx, "val": val_idx, "test": test_idx},
          open(f"{OUT_DIR}/splits.json", "w"))
print(f"PlantVillage: train {len(train_idx)} · val {len(val_idx)} · test {len(test_idx)}")

## 2 · PlantDoc — real field photos of the same crops
2,598 photos from the web/fields, 27 classes; 25 of them match a PlantVillage class. Labels are a bit noisy (it is a real-world dataset). We train on its **train** folder and report accuracy on its **test** folder only.

In [ ]:
PD2PV = {
 "apple scab leaf": "Apple___Apple_scab", "apple leaf": "Apple___healthy", "apple rust leaf": "Apple___Cedar_apple_rust",
 "bell pepper leaf": "Pepper,_bell___healthy", "bell pepper leaf spot": "Pepper,_bell___Bacterial_spot",
 "blueberry leaf": "Blueberry___healthy", "cherry leaf": "Cherry_(including_sour)___healthy",
 "corn gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
 "corn rust leaf": "Corn_(maize)___Common_rust_", "peach leaf": "Peach___healthy",
 "potato leaf early blight": "Potato___Early_blight", "potato leaf late blight": "Potato___Late_blight",
 "raspberry leaf": "Raspberry___healthy", "soyabean leaf": "Soybean___healthy", "squash powdery mildew leaf": "Squash___Powdery_mildew",
 "strawberry leaf": "Strawberry___healthy", "tomato early blight leaf": "Tomato___Early_blight",
 "tomato septoria leaf spot": "Tomato___Septoria_leaf_spot", "tomato leaf": "Tomato___healthy",
 "tomato leaf bacterial spot": "Tomato___Bacterial_spot", "tomato leaf late blight": "Tomato___Late_blight",
 "tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus", "tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
 "tomato mold leaf": "Tomato___Leaf_Mold", "grape leaf": "Grape___healthy", "grape leaf black rot": "Grape___Black_rot",
}
norm = lambda s: re.sub(r"[\s_]+", " ", s).strip().lower()

if FAST_TEST:
    PD_DIR = os.environ.get("FAST_PD_DIR", "")
else:
    PD_DIR = f"{TMP}/PlantDoc-Dataset"
    if not os.path.isdir(f"{PD_DIR}/test"):
        sh(f"git clone --depth 1 -q https://github.com/pratikkayal/PlantDoc-Dataset {PD_DIR}", timeout=900, check=False)

def ok_image(p):
    try:
        with Image.open(p) as im: im.draft("RGB", (256, 256)); im.convert("RGB").load()
        return True
    except Exception: return False

def pd_items(split):
    out, skipped = [], set()
    root = os.path.join(PD_DIR, split)
    if not os.path.isdir(root): return out
    for d in sorted(os.listdir(root)):
        tgt = PD2PV.get(norm(d))
        if tgt is None or tgt not in cls_idx: skipped.add(d); continue
        for f in sorted(os.listdir(os.path.join(root, d))):
            p = os.path.join(root, d, f)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")) and ok_image(p): out.append((p, cls_idx[tgt]))
    if skipped: print(f"  {split}: no PlantVillage match, skipped →", sorted(skipped))
    return out

pd_train_all, pd_test = pd_items("train"), pd_items("test")
HAVE_PD = len(pd_train_all) > 0 and len(pd_test) > 0
random.Random(SEED).shuffle(pd_train_all)
n_pdv = max(1, len(pd_train_all) // 10) if HAVE_PD else 0
pd_val, pd_train = pd_train_all[:n_pdv], pd_train_all[n_pdv:]
print(f"PlantDoc: train {len(pd_train)} · val {len(pd_val)} · test {len(pd_test)}" if HAVE_PD
      else "⚠️ PlantDoc not available — training continues with background swapping only (no real-photo numbers)")
json.dump([[p, y] for p, y in pd_test], open(f"{OUT_DIR}/plantdoc_test.json", "w"))
json.dump([[p, y] for p, y in pd_train[:80]], open(f"{OUT_DIR}/plantdoc_calib.json", "w"))

## 3 · Textures used as new backgrounds (walls, cloth, wood…)

In [ ]:
TEXTURES = []
if not FAST_TEST:
    sh(f"{sys.executable} -c \"import torchvision; torchvision.datasets.DTD('{TMP}/dtd', split='train', download=True)\"", timeout=600, check=False)
    TEXTURES = sorted(glob.glob(f"{TMP}/dtd/**/images/**/*.jpg", recursive=True))
print("background textures:", len(TEXTURES))

## 4 · Leaf cut-out + background swapping (same code as notebook 01 v3)

In [ ]:
SEV = dict(BG_DIST=18, LESION_DIST=22, BORDER=6, KEEP_PIECE=0.15, MILD=0.10, MODERATE=0.25)
GREEN = dict(H_MIN=30, H_MAX=90, S_MIN=35)     # OpenCV hue 0–180 — same "greenish" test as the app

def _lab(im):
    return cv2.cvtColor(im, cv2.COLOR_RGB2LAB).astype(np.float32) * np.array([100/255, 1, 1], np.float32) - np.array([0, 128, 128], np.float32)

def _fill_holes(m):
    h, w = m.shape; pad = np.zeros((h + 2, w + 2), np.uint8); pad[1:-1, 1:-1] = m
    ff = pad.copy(); cv2.floodFill(ff, np.zeros((h + 4, w + 4), np.uint8), (0, 0), 1)
    holes = (ff == 0)[1:-1, 1:-1]
    return m.astype(bool) | holes

def leaf_mask(rgb, cfg=SEV):
    L = _lab(rgb); h, w = L.shape[:2]; b = cfg["BORDER"]
    ring = np.zeros((h, w), bool); ring[:b] = ring[-b:] = True; ring[:, :b] = ring[:, -b:] = True
    bg = np.median(L[ring], 0); d_bg = np.sqrt(((L - bg) ** 2).sum(-1))
    fg = (d_bg > cfg["BG_DIST"]).astype(np.uint8)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(fg, 8)
    if n <= 1: return np.zeros((h, w), bool), d_bg, L
    big = st[1:, cv2.CC_STAT_AREA].max()
    keep = np.isin(lab, [i for i in range(1, n) if st[i, cv2.CC_STAT_AREA] >= cfg["KEEP_PIECE"] * big])
    leaf = _fill_holes(keep.astype(np.uint8))
    if leaf.mean() > 0.97: leaf = np.ones((h, w), bool)
    return leaf, d_bg, L

def severity(rgb224, cfg=SEV):
    leaf, d_bg, L = leaf_mask(rgb224, cfg); h, w = leaf.shape
    if not leaf.any(): return 0.0, "No leaf", leaf, leaf
    core = cv2.erode(leaf.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    hsv = cv2.cvtColor(rgb224, cv2.COLOR_RGB2HSV); H, S, V = [hsv[..., i].astype(int) for i in range(3)]
    greenish = core & (H >= GREEN["H_MIN"]) & (H <= GREEN["H_MAX"]) & (S >= GREEN["S_MIN"])
    ref = np.median(L[greenish], 0) if greenish.sum() > 50 else np.median(L[core], 0)
    shadow = ((V < 110) & (S < 55)) | (V < 45)
    les = core & (np.sqrt(((L - ref) ** 2).sum(-1)) > cfg["LESION_DIST"]) & (d_bg >= cfg["BG_DIST"]) & ~shadow
    les = cv2.morphologyEx(les.astype(np.uint8), cv2.MORPH_OPEN, np.ones((2, 2), np.uint8)).astype(bool)
    frac = float(les.sum()) / max(int(core.sum()), 1)
    return frac, ("Mild" if frac < cfg["MILD"] else "Moderate" if frac < cfg["MODERATE"] else "Severe"), les, leaf

def green_fraction(rgb224):
    """Share of the found 'leaf' that is plant-green. Real leaves: high. Faces, walls, clothes: near 0."""
    leaf, _, _ = leaf_mask(rgb224)
    if leaf.sum() < 0.05 * leaf.size: return 0.0
    hsv = cv2.cvtColor(rgb224, cv2.COLOR_RGB2HSV)
    g = (hsv[..., 0] >= GREEN["H_MIN"]) & (hsv[..., 0] <= GREEN["H_MAX"]) & (hsv[..., 1] >= GREEN["S_MIN"])
    return float((g & leaf).sum()) / float(leaf.sum())

def app_square(path_or_img, size=224):
    """Exactly what the app does: centre square → 512 → 224."""
    im = path_or_img if isinstance(path_or_img, np.ndarray) else np.array(_open(path_or_img))
    s = min(im.shape[:2]); y0 = (im.shape[0] - s) // 2; x0 = (im.shape[1] - s) // 2
    return cv2.resize(cv2.resize(im[y0:y0 + s, x0:x0 + s], (512, 512), interpolation=cv2.INTER_AREA), (size, size), interpolation=cv2.INTER_AREA)

def _open(p, draft=512):
    im = Image.open(p); im.draft("RGB", (draft, draft)); return im.convert("RGB")

PALETTE = np.array([[120,85,60],[90,65,45],[150,120,90],[70,55,40],      # soils
                    [60,110,50],[40,80,35],[100,140,70],                  # grass / foliage
                    [128,128,128],[200,200,200],[240,240,235],[60,60,60], # greys, paper, dark
                    [225,180,150],[160,110,80],                           # hands / skin
                    [150,190,230],[170,130,90],[200,170,120]], np.float32) # sky, wood, sand

def random_background(rng, size=256):
    r = rng.random()
    if r < 0.35 and pd_train:            # blurred real field photo
        im = np.array(_open(pd_train[rng.integers(len(pd_train))][0]))
        im = cv2.resize(im, (size, size)); k = int(rng.choice([7, 11, 15, 21]))
        return cv2.GaussianBlur(im, (k, k), 0)
    if r < 0.55 and TEXTURES:            # wall / fabric / wood texture
        return cv2.resize(np.array(_open(TEXTURES[rng.integers(len(TEXTURES))])), (size, size))
    if r < 0.70:                          # another lab leaf, heavily blurred (leafy clutter)
        im = np.array(_open(pv_all.samples[train_idx[rng.integers(len(train_idx))]][0]))
        return cv2.GaussianBlur(cv2.resize(im, (size, size)), (31, 31), 0)
    c1, c2 = PALETTE[rng.integers(len(PALETTE))], PALETTE[rng.integers(len(PALETTE))]
    t = np.linspace(0, 1, size)[:, None, None] if rng.random() < 0.5 else np.zeros((size, 1, 1))
    bg = c1 * (1 - t) + c2 * t + np.zeros((size, size, 3))
    blotch = cv2.resize(rng.normal(0, 1, (8, 8)).astype(np.float32), (size, size), interpolation=cv2.INTER_CUBIC)[..., None]
    grain = rng.normal(0, 1, (size, size, 1))          # same on all channels → natural texture, not rainbow speckle
    bg = bg * (1 + blotch * rng.uniform(0.03, 0.15)) + grain * rng.uniform(1, 8)
    return np.clip(bg, 0, 255).astype(np.uint8)

def swap_background(img, rng, size=256):
    a = cv2.resize(np.array(img), (size, size))
    leaf, _, _ = leaf_mask(a)
    if leaf.mean() < 0.05 or leaf.mean() > 0.95: return img
    m = cv2.GaussianBlur(leaf.astype(np.float32), (5, 5), 0)
    s = rng.uniform(0.55, 1.0); n = int(size * s)
    ang = rng.uniform(-180, 180); M = cv2.getRotationMatrix2D((size / 2, size / 2), ang, 1.0)
    a = cv2.resize(cv2.warpAffine(a, M, (size, size), borderValue=0), (n, n))
    m = cv2.resize(cv2.warpAffine(m, M, (size, size), borderValue=0), (n, n))[..., None]
    bg = random_background(rng, size).astype(np.float32)
    y, x = rng.integers(0, size - n + 1), rng.integers(0, size - n + 1)
    bg[y:y + n, x:x + n] = bg[y:y + n, x:x + n] * (1 - m) + a * m
    if rng.random() < 0.3:                # cast a shadow across part of the photo
        yy, xx = np.mgrid[0:size, 0:size]; th = rng.uniform(0, np.pi)
        side = (np.cos(th) * (xx - size / 2) + np.sin(th) * (yy - size / 2)) > rng.uniform(-60, 60)
        bg[side] *= rng.uniform(0.45, 0.8)
    return Image.fromarray(np.clip(bg, 0, 255).astype(np.uint8))

class RandomJPEG:
    def __init__(self, p=0.4): self.p = p
    def __call__(self, im):
        if random.random() > self.p: return im
        b = io.BytesIO(); im.save(b, "JPEG", quality=random.randint(25, 85)); b.seek(0); return Image.open(b).convert("RGB")

## 5 · Training data (288 px)

In [ ]:
from torch.utils.data import WeightedRandomSampler
photo_tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.45, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.2),
                      T.TrivialAugmentWide(), T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.25),
                      RandomJPEG(0.4), T.ToTensor()])
eval_tf = T.Compose([T.Resize(int(IMG_SIZE * 1.143)), T.CenterCrop(IMG_SIZE), T.ToTensor()])
BASE = int(IMG_SIZE * 1.12)          # lab photos are prepared at this size before cropping

class TrainSet(Dataset):
    def __init__(self): self.items = [(pv_all.samples[i][0], pv_all.samples[i][1], "pv") for i in train_idx] + \
                                     [(p, y, "pd") for p, y in pd_train] * PLANTDOC_REPEAT
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        p, y, src = self.items[k]; rng = np.random.default_rng(random.getrandbits(32))
        im = _open(p, 640)
        if src == "pv":
            im = im.resize((BASE, BASE))
            if rng.random() < BG_SWAP_PROB: im = swap_background(im, rng, size=BASE)
        return photo_tf(im), y

class AppSet(Dataset):          # photos preprocessed exactly like the app (centre square → 512 → IMG_SIZE)
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        p, y = self.items[k]
        return torch.from_numpy(app_square(p, IMG_SIZE)).permute(2, 0, 1).float() / 255, y

train_set = TrainSet()
# balanced sampling: each photo weighted by 1/sqrt(class size) → rare classes appear ~2-4× more often than before
counts = np.bincount([y for _, y, _ in train_set.items], minlength=len(classes)).astype(np.float64)
w = np.array([1.0 / np.sqrt(counts[y]) for _, y, _ in train_set.items])
sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(train_set), replacement=True)
workers = 0 if FAST_TEST else 3
# fresh worker processes every epoch (no persistent_workers): memory they build up is returned to the system each epoch
train_dl = DataLoader(train_set, BATCH, sampler=sampler, num_workers=workers, pin_memory=True, drop_last=True)
pv_eval = ImageFolder(DATA_DIR, transform=eval_tf)
val_dl   = DataLoader(Subset(pv_eval, val_idx),  BATCH, num_workers=workers)
test_dl  = DataLoader(Subset(pv_eval, test_idx), BATCH, num_workers=workers)
pdval_dl = DataLoader(AppSet(pd_val),  BATCH, num_workers=workers) if HAVE_PD else None
pdtest_dl= DataLoader(AppSet(pd_test), BATCH, num_workers=workers) if HAVE_PD else None
print("training images per epoch:", len(train_set), "· smallest class", int(counts.min()), "· largest", int(counts.max()))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 6, figsize=(15, 8))
for a in ax.flat:
    x, y = train_set[random.randrange(len(train_set))]
    a.imshow(x.permute(1, 2, 0).numpy()); a.set_title(classes[y].replace("___", "\n")[:34], fontsize=7); a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_examples.png", dpi=100); plt.close()

## 6 · Model and training (EfficientNet-B2, CutMix/MixUp, EMA, cosine schedule)

In [ ]:
MODEL_SRC = """
import torch, torch.nn as nn, torchvision
class Classifier(nn.Module):
    # Normalisation + EfficientNet-B2 + (softmax when export_mode) -> the .pte takes 0-1 RGB and returns probabilities.
    def __init__(self, n_classes, pretrained=True):
        super().__init__()
        w = torchvision.models.EfficientNet_B2_Weights.IMAGENET1K_V1 if pretrained else None
        self.net = torchvision.models.efficientnet_b2(weights=w)
        self.net.classifier[1] = nn.Linear(self.net.classifier[1].in_features, n_classes)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.export_mode = False
    def forward(self, x):
        logits = self.net((x - self.mean) / self.std)
        return torch.softmax(logits, dim=1) if self.export_mode else logits
"""
open(f"{WORK}/vision_model_v4.py", "w").write(MODEL_SRC)
sys.path.insert(0, WORK)
from vision_model_v4 import Classifier
import torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torchvision.transforms import v2 as T2

base = Classifier(len(classes), pretrained=not FAST_TEST).to(device)
model = nn.DataParallel(base) if N_GPU > 1 else base
ema = AveragedModel(base, multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY), use_buffers=True)
opt = torch.optim.AdamW(base.parameters(), lr=LR, weight_decay=1e-2)
steps = EPOCHS * len(train_dl); warm = WARMUP_EPOCHS * len(train_dl)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                          0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, steps - warm))))
mixer = T2.RandomChoice([T2.CutMix(num_classes=len(classes)), T2.MixUp(num_classes=len(classes))])
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
crit_hard = nn.CrossEntropyLoss(label_smoothing=0.1)
crit_soft = nn.CrossEntropyLoss()

@torch.no_grad()
def predict(net, dl, tta=False):
    net.eval(); P, Y = [], []
    for x, y in dl:
        x = x.to(device)
        p = torch.softmax(net(x).float(), 1)
        if tta: p = 0.5 * (p + torch.softmax(net(torch.flip(x, dims=[3])).float(), 1))
        P.append(p.cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y)
acc = lambda P, Y: (P.argmax(1) == Y).float().mean().item()

import gc
best = -1.0
if RESUME:
    shutil.copy(RESUME[0], f"{OUT_DIR}/best.pt"); print("using", RESUME[0], "— training skipped")
for ep in range(0 if RESUME else EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0; n = 0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        mixed = random.random() < MIX_PROB
        if mixed: x, y_soft = mixer(x, y)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=use_amp):
            out = model(x)
            loss = crit_soft(out, y_soft) if mixed else crit_hard(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update_parameters(base)
        tot += loss.item() * x.size(0); n += x.size(0)
    lab_acc = acc(*predict(ema.module, val_dl))
    field_acc = acc(*predict(ema.module, pdval_dl)) if HAVE_PD else float("nan")
    score = (lab_acc + field_acc) / 2 if HAVE_PD else lab_acc
    print(f"epoch {ep+1}/{EPOCHS} · loss {tot/n:.4f} · lab val {lab_acc:.4f} · real-photo val {field_acc:.4f} (EMA) · {time.time()-t0:.0f}s · {ram()}", flush=True)
    if score >= best:
        best = score; torch.save(ema.module.state_dict(), f"{OUT_DIR}/best.pt"); print("  ↳ saved best.pt", flush=True)
    gc.collect()
# free the training data loaders, optimiser and model before testing
del train_dl, model, opt, ema, base, scaler
gc.collect(); torch.cuda.empty_cache() if device == "cuda" else None
print("after training:", ram())
final = Classifier(len(classes), pretrained=False).to(device)
final.load_state_dict(torch.load(f"{OUT_DIR}/best.pt", map_location=device))

## 7 · Test results (lab + real photos, with and without the mirror-image average)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
crop_of = [l["crop"] for l in labels]
crop_mask = torch.tensor([[crop_of[j] == crop_of[i] for j in range(len(classes))] for i in range(len(classes))])
def field_report(P, Y):
    crop_known = (P * crop_mask[Y]).argmax(1)
    return {"n": len(Y), "accuracy_any_crop": acc(P, Y), "top3_any_crop": (P.topk(3, 1).indices == Y[:, None]).any(1).float().mean().item(),
            "accuracy_crop_known": (crop_known == Y).float().mean().item(),
            "wrong_crop_rate": float(np.mean([crop_of[p] != crop_of[y] for p, y in zip(P.argmax(1).tolist(), Y.tolist())]))}
PAIRS = [("Tomato___Septoria_leaf_spot", "Tomato___Leaf_Mold"), ("Peach___healthy", "Orange___Haunglongbing_(Citrus_greening)"),
         ("Peach___Bacterial_spot", "Orange___Haunglongbing_(Citrus_greening)"), ("Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus"),
         ("Tomato___Early_blight", "Tomato___Target_Spot"), ("Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Northern_Leaf_Blight")]
def pair_errors(P, Y):
    pr = P.argmax(1)
    out = {}
    for a, b in PAIRS:
        if a in cls_idx and b in cls_idx:
            ia, ib = cls_idx[a], cls_idx[b]
            out[f"{a} → {b}"] = int(((Y == ia) & (pr == ib)).sum()); out[f"{b} → {a}"] = int(((Y == ib) & (pr == ia)).sum())
    return out

metrics = {"version": "v4", "arch": "efficientnet_b2", "input_size": IMG_SIZE}
for tta in (False, True):
    P, Y = predict(final, test_dl, tta)
    k = "tta" if tta else "single"
    metrics[f"lab_{k}"] = {"accuracy": acc(P, Y), "macro_f1": f1_score(Y, P.argmax(1), average="macro"), "n": len(Y),
                           "look_alike_errors": pair_errors(P, Y)}
    if HAVE_PD:
        Pf, Yf = predict(final, pdtest_dl, tta)
        metrics[f"real_{k}"] = {**field_report(Pf, Yf), "look_alike_errors": pair_errors(Pf, Yf)}
    if tta:
        cm = confusion_matrix(Y, P.argmax(1), labels=list(range(len(classes))))
        plt.figure(figsize=(12, 10)); plt.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues")
        plt.title("v4 confusion matrix — PlantVillage test (row-normalised, mirror-averaged)"); plt.colorbar()
        plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrix_v4.png", dpi=150); plt.close()
        metrics["per_class_lab_tta"] = classification_report(Y, P.argmax(1), labels=list(range(len(classes))), target_names=classes,
                                                             output_dict=True, zero_division=0)
        # "Not sure" cut-off on v4 probabilities: ~7.5 % of correct real-photo answers fall below it
        Pv, Yv = predict(final, pdval_dl, True) if HAVE_PD else predict(final, val_dl, True)
        corr = Pv.max(1).values[Pv.argmax(1) == Yv].numpy()
        MIN_CONF = float(np.clip(round(np.percentile(corr, 7.5) / 0.05) * 0.05, 0.35, 0.6)) if len(corr) else 0.45
for k in ("single", "tta"):
    print(f"LAB  {k:6s}: accuracy {metrics[f'lab_{k}']['accuracy']:.4f} · macro-F1 {metrics[f'lab_{k}']['macro_f1']:.4f}")
    if HAVE_PD:
        r = metrics[f"real_{k}"]
        print(f"REAL {k:6s}: any crop {r['accuracy_any_crop']:.3f} · top-3 {r['top3_any_crop']:.3f} · crop known {r['accuracy_crop_known']:.3f} · wrong crop {r['wrong_crop_rate']:.3f}")
print("look-alike mistakes (lab test, mirror-averaged):", {k: v for k, v in metrics["lab_tta"]["look_alike_errors"].items() if v})
print("min_confidence:", MIN_CONF)
json.dump(metrics, open(f"{OUT_DIR}/metrics_v4.json", "w"), indent=1, default=float)

## 8 · Export to ExecuTorch (FP32 and INT8), test both on the CPU, compare with v3, pick the one for the app

In [ ]:
PINS = "executorch==1.6.0 torch==2.15.0 torchvision==0.29.2 torchao==0.18.0"
if not os.path.exists(f"{ENV_DIR}/bin/python"):
    sh(f"{sys.executable} -m pip install -q virtualenv"); sh(f"{sys.executable} -m virtualenv -q {ENV_DIR}")
pip = f"{ENV_DIR}/bin/pip install -q --default-timeout=120 --retries 10"
if not FAST_TEST:
    if subprocess.run(f"{pip} --extra-index-url https://download.pytorch.org/whl/cpu {PINS}", shell=True).returncode != 0:
        sh(f"{pip} {PINS}")
sh(f"{ENV_DIR}/bin/python -c \"import torch, executorch; print('export env: torch', torch.__version__)\"")

EXPORT_SRC = r"""
import sys, os, json, time, argparse
import numpy as np, torch
from PIL import Image
from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.runtime import Runtime
ap = argparse.ArgumentParser(); ap.add_argument("--work"); ap.add_argument("--out"); ap.add_argument("--size", type=int)
ap.add_argument("--v3", default=""); ap.add_argument("--n_lab", type=int, default=1500); a = ap.parse_args()
sys.path.insert(0, a.work)
from vision_model_v4 import Classifier

def app_tensor(p, size):   # centre square -> 512 -> size, exactly like the app
    im = Image.open(p); im.draft("RGB", (512, 512)); im = im.convert("RGB")
    w, h = im.size; s = min(w, h); im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    im = im.resize((512, 512), Image.BOX).resize((size, size), Image.BOX)
    return torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1).float().div(255)[None]

sp = json.load(open(f"{a.out}/splits.json")); classes = sp["classes"]
m = Classifier(len(classes), pretrained=False); m.load_state_dict(torch.load(f"{a.out}/best.pt", map_location="cpu")); m.export_mode = True; m.eval()
example = (torch.rand(1, 3, a.size, a.size),)
def save(model, path):
    prog = to_edge_transform_and_lower(export(model, example), partitioner=[XnnpackPartitioner()]).to_executorch()
    open(path, "wb").write(prog.buffer); print(f"saved {os.path.basename(path)} ({len(prog.buffer)/1e6:.1f} MB)", flush=True)
save(m, f"{a.out}/crop_disease_v4_fp32.pte")
from torchvision.datasets import ImageFolder
ds = ImageFolder(sp["data_dir"])
cal = [ds.samples[i][0] for i in sp["train"][:250]] + [p for p, _ in json.load(open(f"{a.out}/plantdoc_calib.json"))]
prepared = prepare_pt2e(export(m, example).module(), XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_per_channel=True)))
with torch.no_grad():
    for p in cal: prepared(app_tensor(p, a.size))
save(convert_pt2e(prepared), f"{a.out}/crop_disease_v4_int8.pte")

lab = [(ds.samples[i][0], ds.samples[i][1]) for i in sp["test"][:a.n_lab]]
real = [tuple(x) for x in json.load(open(f"{a.out}/plantdoc_test.json"))]
def run(path, items, size, tta):
    meth = Runtime.get().load_program(path).load_method("forward"); ok, ms = 0, []
    for p, y in items:
        x = app_tensor(p, size); t = time.perf_counter(); pr = meth.execute([x])[0]
        if tta: pr = 0.5 * (pr + meth.execute([torch.flip(x, dims=[3])])[0])
        ms.append(time.perf_counter() - t); ok += int(pr.argmax(1).item() == y)
    return {"accuracy": ok / max(1, len(items)), "n": len(items), "median_ms_notebook_cpu": 1000 * float(np.median(ms)) if ms else None}
res = {}
for name in ("fp32", "int8"):
    path = f"{a.out}/crop_disease_v4_{name}.pte"
    res[f"v4_{name}"] = {"size_mb": os.path.getsize(path) / 1e6, "lab_tta": run(path, lab, a.size, True), "real_tta": run(path, real, a.size, True) if real else None}
    print(name, json.dumps(res[f"v4_{name}"]), flush=True)
if a.v3 and os.path.exists(f"{a.v3}/crop_disease_int8.pte"):
    res["v3_int8"] = {"lab": run(f"{a.v3}/crop_disease_int8.pte", lab, 224, False), "real": run(f"{a.v3}/crop_disease_int8.pte", real, 224, False) if real else None}
    print("v3", json.dumps(res["v3_int8"]), flush=True)
json.dump(res, open(f"{a.out}/pte_metrics_v4.json", "w"), indent=1)
"""
open(f"{WORK}/export_v4.py", "w").write(EXPORT_SRC)
sh(f"{ENV_DIR}/bin/python {WORK}/export_v4.py --work {WORK} --out {OUT_DIR} --size {IMG_SIZE} --n_lab {60 if FAST_TEST else 1500}"
   + (f" --v3 '{V3}'" if V3 else ""))

pm = json.load(open(f"{OUT_DIR}/pte_metrics_v4.json")); metrics.update(pm)
f32, i8 = pm["v4_fp32"], pm["v4_int8"]
def sc(r): return r["lab_tta"]["accuracy"] + (r["real_tta"]["accuracy"] if r["real_tta"] else 0)
use_int8 = (i8["lab_tta"]["accuracy"] >= f32["lab_tta"]["accuracy"] - 0.005 and
            (not f32["real_tta"] or i8["real_tta"]["accuracy"] >= f32["real_tta"]["accuracy"] - 0.005))
CHOSEN = "crop_disease_v4_int8.pte" if use_int8 else "crop_disease_v4_fp32.pte"
vision_config = {"version": "v4", "disease_model": CHOSEN, "input_size": IMG_SIZE, "tta_flip": True, "min_confidence": MIN_CONF,
                 "arch": "efficientnet_b2"}
json.dump(vision_config, open(f"{OUT_DIR}/vision_config.json", "w"), indent=1)
json.dump(metrics, open(f"{OUT_DIR}/metrics_v4.json", "w"), indent=1, default=float)
print("\nChosen for the app:", CHOSEN, "(INT8 within 0.5 points of FP32)" if use_int8 else "(INT8 lost accuracy → FP32)")

## 9 · Package

In [ ]:
pkg = f"{WORK}/ps1_vision_outputs_v4"; os.makedirs(pkg, exist_ok=True)
for f in [CHOSEN, "labels.json", "vision_config.json", "metrics_v4.json", "confusion_matrix_v4.png", "training_examples.png"]:
    shutil.copy(f"{OUT_DIR}/{f}", pkg)
print("zipped →", shutil.make_archive(pkg, "zip", pkg))
c = pm[f"v4_{'int8' if use_int8 else 'fp32'}"]
print(f"\nv4 ({CHOSEN}, {c['size_mb']:.1f} MB): lab {c['lab_tta']['accuracy']:.4f}"
      + (f" · REAL photos {c['real_tta']['accuracy']:.4f}" if c["real_tta"] else "") + f" · {c['lab_tta']['median_ms_notebook_cpu']:.0f} ms/photo (2 passes, notebook CPU)")
if "v3_int8" in pm:
    v = pm["v3_int8"]; print(f"v3 (for comparison): lab {v['lab']['accuracy']:.4f}" + (f" · REAL photos {v['real']['accuracy']:.4f}" if v["real"] else ""))
print("look-alike mistakes, v4 lab test:", {k: v for k, v in metrics["lab_tta"]["look_alike_errors"].items() if v})
if "real_tta" in metrics: print("look-alike mistakes, v4 real photos:", {k: v for k, v in metrics["real_tta"]["look_alike_errors"].items() if v})
print("min_confidence:", MIN_CONF)